In [1]:
import requests
base = "https://www.ncbi.nlm.nih.gov/research/pubtator3-api"
r = requests.get(f"{base}/entity/autocomplete/", params={"query": "colorectal cancer", "limit": 5})
print(r.json())   

[{'_id': '@DISEASE_Colorectal_Neoplasms', 'biotype': 'disease', 'db_id': 'D015179', 'db': 'ncbi_mesh', 'name': 'Colorectal Neoplasms', 'match': 'Multiple matches'}, {'_id': '@DISEASE_Colorectal_Neoplasms_Hereditary_Nonpolyposis', 'biotype': 'disease', 'db_id': 'D003123', 'db': 'ncbi_mesh', 'name': 'Colorectal Neoplasms Hereditary Nonpolyposis', 'match': 'Multiple matches'}, {'_id': '@DISEASE_Colitis_Associated_Neoplasms', 'biotype': 'disease', 'db_id': 'D000083023', 'db': 'ncbi_mesh', 'name': 'Colitis-Associated Neoplasms', 'match': 'Multiple matches'}, {'_id': '@DISEASE_Oligodontia_Colorectal_Cancer_Syndrome', 'biotype': 'disease', 'db_id': 'C563898', 'db': 'ncbi_mesh', 'name': 'Oligodontia-Colorectal Cancer Syndrome', 'match': 'Multiple matches'}, {'_id': '@DISEASE_Lynch_Syndrome_II', 'biotype': 'disease', 'db_id': 'D055847', 'db': 'ncbi_mesh', 'name': 'Lynch Syndrome II', 'match': 'Multiple matches'}]


In [2]:
r = requests.get(f"{base}/relations",
                 params={"e1": "@DISEASE_Colorectal_Neoplasms", "type": "treat", "e2": "chemical"})
print(r.json())

[{'type': 'treat', 'source': '@CHEMICAL_Fluorouracil', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 6797}, {'type': 'treat', 'source': '@CHEMICAL_Oxaliplatin', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 4809}, {'type': 'treat', 'source': '@CHEMICAL_Irinotecan', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 3446}, {'type': 'treat', 'source': '@CHEMICAL_Bevacizumab', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 3072}, {'type': 'treat', 'source': '@CHEMICAL_Cetuximab', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 2639}, {'type': 'treat', 'source': '@CHEMICAL_Leucovorin', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 1979}, {'type': 'treat', 'source': '@CHEMICAL_Capecitabine', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 1358}, {'type': 'treat', 'source': '@CHEMICAL_Folfox_protocol', 'target': '@DISEASE_Colorectal_Neoplasms', 'publications': 1147}, {'type': 'treat', 'source': '@CHEMICAL_Panitu

In [3]:
import os, requests
os.makedirs("data", exist_ok=True)
url = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/relation2pubtator3.gz"
path = "data/relation2pubtator3.gz"

with requests.get(url, stream=True) as resp:
    resp.raise_for_status()
    with open(path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)

print(round(os.path.getsize(path) / 1e6), "MB downloaded")

297 MB downloaded


In [4]:
import gzip
with gzip.open(path, "rt", encoding="utf-8") as f:
    for _ in range(10):
        print(f.readline().rstrip())

35378878	associate	Chemical|MESH:D003911	Disease|MESH:D005334
35378880	associate	Chemical|MESH:D010866	Chemical|MESH:D011108
35378880	treat	Chemical|MESH:D010866	Disease|MESH:D009181
35378880	treat	Chemical|MESH:D010866	Disease|MESH:D060585
35378880	treat	Chemical|MESH:D011108	Disease|MESH:D060585
35378888	associate	Chemical|MESH:D002245	Chemical|MESH:D002482
35378888	associate	Chemical|MESH:D002245	Chemical|MESH:D011108
35378891	treat	Chemical|MESH:D002439	Disease|MESH:D014435
35378891	treat	Chemical|MESH:D019980	Disease|MESH:D014435
35378891	treat	Chemical|MESH:D000077731	Disease|MESH:D014435


In [5]:
import duckdb
con = duckdb.connect("data/pubtator.duckdb")

con.execute("""
CREATE OR REPLACE TABLE crc_relations AS
SELECT *
FROM read_csv('data/relation2pubtator3.gz',
              delim = '\t', header = false,
              columns = {'pmid': 'BIGINT', 'rel_type': 'VARCHAR',
                         'entity1': 'VARCHAR', 'entity2': 'VARCHAR'})
WHERE entity1 = 'Disease|MESH:D015179'
   OR entity2 = 'Disease|MESH:D015179'
""")

con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pmid) AS n_articles FROM crc_relations").show()

┌────────┬────────────┐
│ n_rows │ n_articles │
│ int64  │   int64    │
├────────┼────────────┤
│ 233074 │     106817 │
└────────┴────────────┘



In [6]:
con.sql("""
SELECT rel_type,
       CASE WHEN entity1 = 'Disease|MESH:D015179' THEN entity2 ELSE entity1 END AS partner,
       COUNT(DISTINCT pmid) AS n_articles
FROM crc_relations
GROUP BY 1, 2
ORDER BY n_articles DESC
LIMIT 20
""").show()

┌───────────┬──────────────────────────┬────────────┐
│ rel_type  │         partner          │ n_articles │
│  varchar  │         varchar          │   int64    │
├───────────┼──────────────────────────┼────────────┤
│ treat     │ Chemical|MESH:D005472    │       7029 │
│ treat     │ Chemical|MESH:D000077150 │       5046 │
│ associate │ Gene|3845                │       4620 │
│ treat     │ Chemical|MESH:D000077146 │       3541 │
│ associate │ Gene|7157                │       3264 │
│ treat     │ Chemical|MESH:D000068258 │       3236 │
│ associate │ Gene|1956                │       2762 │
│ treat     │ Chemical|MESH:D000068818 │       2760 │
│ associate │ Gene|673                 │       2677 │
│ associate │ Gene|1499                │       2553 │
│ treat     │ Chemical|MESH:D002955    │       2004 │
│ associate │ Gene|207                 │       1754 │
│ associate │ Gene|1084                │       1567 │
│ treat     │ Chemical|MESH:D000069287 │       1407 │
│ associate │ Gene|324      

In [8]:
import requests, time, pandas as pd

ESUM = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
base_params = {"db": "pubmed", "retmode": "json", "email": "aaronpong21@gmail.com"}
# base_params["api_key"] = "YOUR_NCBI_KEY"
pause = 0.35   # change to 0.11 if you add the API key

pmids = con.sql("SELECT DISTINCT pmid FROM crc_relations").df()["pmid"].astype(str).tolist()

rows, bad = [], 0
for i in range(0, len(pmids), 200):
    batch = pmids[i:i + 200]
    for attempt in range(3):
        try:
            resp = requests.post(ESUM, data={**base_params, "id": ",".join(batch)}, timeout=60)
            resp.raise_for_status()
            result = resp.json()["result"]
            break
        except Exception as e:
            print(f"Batch {i} failed ({e}), retrying...")
            time.sleep(5)
    else:
        print(f"Skipping batch {i}")
        continue
    for uid in result.get("uids", []):
        doc = result[uid]
        if "error" in doc:
            bad += 1
            continue
        rows.append({"pmid": int(uid), "year": doc.get("pubdate", "")[:4]})
    if i % 10000 == 0:
        print(f"{i:,} of {len(pmids):,} done")
    time.sleep(pause)

print(f"Done: {len(rows):,} articles with dates, {bad:,} IDs PubMed couldn't return")

years = pd.DataFrame(rows)
years["year"] = pd.to_numeric(years["year"], errors="coerce").astype("Int64")
con.execute("CREATE OR REPLACE TABLE pmid_years AS SELECT * FROM years")

con.sql("""
SELECT COUNT(*) AS n_articles, COUNT(year) AS with_year,
       MIN(year) AS first_year, MAX(year) AS last_year
FROM pmid_years
""").show()

0 of 106,817 done
10,000 of 106,817 done
20,000 of 106,817 done
30,000 of 106,817 done
40,000 of 106,817 done
50,000 of 106,817 done
60,000 of 106,817 done
70,000 of 106,817 done
80,000 of 106,817 done
90,000 of 106,817 done
100,000 of 106,817 done
Done: 106,771 articles with dates, 46 IDs PubMed couldn't return
┌────────────┬───────────┬────────────┬───────────┐
│ n_articles │ with_year │ first_year │ last_year │
│   int64    │   int64   │   int64    │   int64   │
├────────────┼───────────┼────────────┼───────────┤
│     106771 │    106771 │       1948 │      2027 │
└────────────┴───────────┴────────────┴───────────┘



In [9]:
counts = con.sql("""
SELECT r.rel_type,
       CASE WHEN r.entity1 = 'Disease|MESH:D015179' THEN r.entity2 ELSE r.entity1 END AS partner,
       y.year,
       COUNT(DISTINCT r.pmid) AS n
FROM crc_relations r
JOIN pmid_years y USING (pmid)
WHERE y.year BETWEEN 2015 AND 2025
GROUP BY 1, 2, 3
""").df()

field = con.sql("""
SELECT y.year, COUNT(DISTINCT r.pmid) AS n
FROM crc_relations r
JOIN pmid_years y USING (pmid)
WHERE y.year BETWEEN 2015 AND 2025
GROUP BY 1 ORDER BY 1
""").df()

In [10]:
import numpy as np
from sklearn.linear_model import LinearRegression

years_grid = np.arange(2015, 2026)
X = years_grid.reshape(-1, 1)

# Overall colorectal cancer growth rate (the comparison baseline)
field_y = np.log1p(field.set_index("year")["n"].reindex(years_grid, fill_value=0).to_numpy())
field_growth = np.expm1(LinearRegression().fit(X, field_y).coef_[0])
print(f"Colorectal cancer literature overall: {field_growth:+.1%} per year")

# One row per relationship, one column per year; missing years become 0
wide = counts.pivot_table(index=["rel_type", "partner"], columns="year",
                          values="n", fill_value=0)
wide = wide.reindex(columns=years_grid, fill_value=0)
wide = wide[wide.sum(axis=1) >= 50]   # need enough papers to trust a trend

# Fit every relationship's trend at once
model = LinearRegression().fit(X, np.log1p(wide.to_numpy()).T)

trends = wide.reset_index()[["rel_type", "partner"]].copy()
trends["total_2015_2025"] = wide.sum(axis=1).to_numpy()
trends["recent_2023_2025"] = wide[[2023, 2024, 2025]].sum(axis=1).to_numpy()
trends["growth_per_year"] = np.expm1(model.coef_[:, 0])
trends["excess_growth"] = trends["growth_per_year"] - field_growth
trends = trends.sort_values("excess_growth", ascending=False).reset_index(drop=True)

con.execute("CREATE OR REPLACE TABLE trends AS SELECT * FROM trends")
print(f"{len(trends):,} relationships with 50+ papers")
trends.head(20)

Colorectal cancer literature overall: +4.6% per year
304 relationships with 50+ papers


year,rel_type,partner,total_2015_2025,recent_2023_2025,growth_per_year,excess_growth
0,associate,Chemical|MESH:C005955,170.0,93.0,0.544547,0.498551
1,associate,Gene|2879,95.0,78.0,0.493661,0.447665
2,treat,Chemical|MESH:C000591844,152.0,107.0,0.469467,0.423471
3,associate,Chemical|MESH:C010223,94.0,44.0,0.462750,0.416754
4,treat,Chemical|MESH:C000601108,121.0,63.0,0.442138,0.396142
5,associate,Gene|56339,74.0,43.0,0.434171,0.388175
6,associate,Gene|60533,140.0,76.0,0.407957,0.361961
7,associate,Gene|18566,117.0,63.0,0.406405,0.360409
8,associate,Gene|340061,69.0,53.0,0.357451,0.311455
9,treat,Chemical|MESH:D000074324,93.0,41.0,0.355279,0.309283


In [12]:
def mesh_label_v2(mesh_id):
    # 1) NLM MeSH JSON page
    for attempt in range(3):
        try:
            r = requests.get(f"https://id.nlm.nih.gov/mesh/{mesh_id}.json", timeout=30)
            if r.status_code == 404:
                break
            r.raise_for_status()
            lab = r.json().get("label")
            if isinstance(lab, list) and lab:
                lab = lab[0]
            if isinstance(lab, dict):
                return lab.get("@value", mesh_id)
            if isinstance(lab, str):
                return lab
            break
        except Exception:
            time.sleep(2)
    # 2) Fallback: NCBI's MeSH database
    try:
        s = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                         params={"db": "mesh", "term": mesh_id, "retmode": "json",
                                 "email": "aaronpong21@gmail.com"}, timeout=30).json()
        ids = s["esearchresult"]["idlist"]
        if ids:
            d = requests.get(ESUM, params={"db": "mesh", "id": ids[0], "retmode": "json"},
                             timeout=30).json()["result"][ids[0]]
            return d.get("ds_meshterms", [mesh_id])[0]
    except Exception:
        pass
    return mesh_id

missing = [m for m, name in mesh_names.items() if name == m]
print(f"Retrying {len(missing)} chemical IDs...")
for m in missing:
    mesh_names[m] = mesh_label_v2(m)
    time.sleep(0.35)

still = [m for m in missing if mesh_names[m] == m]
print(f"Still unresolved: {len(still)}", still[:10])

trends["partner_name"] = trends["partner"].map(label)
con.execute("CREATE OR REPLACE TABLE trends AS SELECT * FROM trends")

show = trends[["rel_type", "kind", "partner_name", "total_2015_2025", "recent_2023_2025"]].copy()
show[["total_2015_2025", "recent_2023_2025"]] = show[["total_2015_2025", "recent_2023_2025"]].astype(int)
show["growth_%/yr"] = (trends["growth_per_year"] * 100).round(1)
show["excess_%/yr"] = (trends["excess_growth"] * 100).round(1)
show.columns.name = None
show.head(25)

Retrying 47 chemical IDs...
Still unresolved: 0 []


,rel_type,kind,partner_name,total_2015_2025,recent_2023_2025,growth_%/yr,excess_%/yr
0,associate,Chemical,[OBSOLETE] 6-methyladenine,170,93,54.5,49.9
1,associate,Gene,GPX4,95,78,49.4,44.8
2,treat,Chemical,HMPL-013,152,107,46.9,42.3
3,associate,Chemical,N-methyladenosine,94,44,46.3,41.7
4,treat,Chemical,encorafenib,121,63,44.2,39.6
5,associate,Gene,METTL3,74,43,43.4,38.8
6,associate,Gene,Cd274 (Mus musculus),140,76,40.8,36.2
7,associate,Gene,Pdcd1 (Mus musculus),117,63,40.6,36.0
8,associate,Gene,STING1,69,53,35.7,31.1
9,treat,Chemical,Ipilimumab,93,41,35.5,30.9


In [13]:
synonyms = {
    "6-methyladenine": "N6-methyladenosine (m6A)",
    "N-methyladenosine": "N6-methyladenosine (m6A)",
    "HMPL-013": "fruquintinib (HMPL-013)",
}

def canonical(row):
    name = row["partner_name"].replace("[OBSOLETE] ", "")
    if row["kind"] == "Gene":
        name = name.split(" (")[0].upper()   # mouse Cd274 -> CD274
    return synonyms.get(name, name)

trends["canonical"] = trends.apply(canonical, axis=1)
partner_map = trends[["partner", "kind", "canonical"]].drop_duplicates()
con.execute("CREATE OR REPLACE TABLE partner_map AS SELECT * FROM partner_map")

In [14]:
counts2 = con.sql("""
SELECT r.rel_type, m.kind, m.canonical, y.year, COUNT(DISTINCT r.pmid) AS n
FROM crc_relations r
JOIN partner_map m
  ON m.partner = CASE WHEN r.entity1 = 'Disease|MESH:D015179' THEN r.entity2 ELSE r.entity1 END
JOIN pmid_years y USING (pmid)
WHERE y.year BETWEEN 2015 AND 2025
GROUP BY 1, 2, 3, 4
""").df()

wide2 = counts2.pivot_table(index=["rel_type", "kind", "canonical"], columns="year",
                            values="n", fill_value=0)
wide2 = wide2.reindex(columns=years_grid, fill_value=0)
wide2 = wide2[wide2.sum(axis=1) >= 50]
model2 = LinearRegression().fit(X, np.log1p(wide2.to_numpy()).T)

final = wide2.reset_index()[["rel_type", "kind", "canonical"]].copy()
final["total_2015_2025"] = wide2.sum(axis=1).astype(int).to_numpy()
final["recent_2023_2025"] = wide2[[2023, 2024, 2025]].sum(axis=1).astype(int).to_numpy()
final["growth_pct"] = (np.expm1(model2.coef_[:, 0]) * 100).round(1)
final["excess_pct"] = (final["growth_pct"] - field_growth * 100).round(1)
final = final.sort_values("excess_pct", ascending=False).reset_index(drop=True)
final.columns.name = None
con.execute("CREATE OR REPLACE TABLE trends_merged AS SELECT * FROM final")

# Final set to summarize: fast-growing AND enough recent papers to summarize
selected = final[final["recent_2023_2025"] >= 30].head(15)
con.execute("CREATE OR REPLACE TABLE selected AS SELECT * FROM selected")
selected

,rel_type,kind,canonical,total_2015_2025,recent_2023_2025,growth_pct,excess_pct
0,associate,Chemical,N6-methyladenosine (m6A),210,111,58.7,54.1
1,associate,Gene,GPX4,95,78,49.4,44.8
2,treat,Chemical,fruquintinib (HMPL-013),152,107,46.9,42.3
3,treat,Chemical,encorafenib,121,63,44.2,39.6
4,associate,Gene,METTL3,74,43,43.4,38.8
5,associate,Gene,STING1,69,53,35.7,31.1
6,treat,Chemical,Ipilimumab,93,41,35.5,30.9
7,treat,Chemical,Nivolumab,178,75,34.2,29.6
8,associate,Gene,PDCD1,612,307,33.5,28.9
9,associate,Chemical,Cholesterol,84,47,32.6,28.0


In [20]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
print("Claude key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

Claude key loaded: True


In [21]:
import time, pandas as pd
from Bio import Entrez
Entrez.email = "aaronpong21@gmail.com"

con.execute("""
CREATE OR REPLACE TABLE topic_pmids AS
SELECT DISTINCT s.canonical, s.rel_type, r.pmid, y.year
FROM selected s
JOIN partner_map m ON m.canonical = s.canonical
JOIN crc_relations r
  ON r.rel_type = s.rel_type
 AND m.partner = CASE WHEN r.entity1 = 'Disease|MESH:D015179' THEN r.entity2 ELSE r.entity1 END
JOIN pmid_years y ON y.pmid = r.pmid
WHERE y.year BETWEEN 2023 AND 2025
""")
recent_pmids = con.sql("SELECT DISTINCT pmid FROM topic_pmids").df()["pmid"].astype(str).tolist()
print(f"{len(recent_pmids):,} recent articles to fetch")

arts = []
for i in range(0, len(recent_pmids), 200):
    handle = Entrez.efetch(db="pubmed", id=",".join(recent_pmids[i:i + 200]), retmode="xml")
    recs = Entrez.read(handle)
    handle.close()
    for art in recs["PubmedArticle"]:
        cit = art["MedlineCitation"]
        a = cit["Article"]
        abstract = " ".join(str(t) for t in a.get("Abstract", {}).get("AbstractText", []))
        arts.append({"pmid": int(cit["PMID"]), "title": str(a.get("ArticleTitle", "")),
                     "abstract": abstract})
    print(f"{min(i + 200, len(recent_pmids)):,} of {len(recent_pmids):,}")
    time.sleep(0.35)

articles = pd.DataFrame(arts)
articles = articles[articles["abstract"].str.len() > 200]
con.execute("CREATE OR REPLACE TABLE articles AS SELECT * FROM articles")
print(f"{len(articles):,} articles with usable abstracts")

1,382 recent articles to fetch
200 of 1,382
400 of 1,382
600 of 1,382
800 of 1,382
1,000 of 1,382
1,200 of 1,382
1,382 of 1,382
1,332 articles with usable abstracts


In [22]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"
SYSTEM = (
    "You summarize recent biomedical research for a non-specialist audience. "
    "Use ONLY the abstracts provided; do not add outside knowledge. "
    "Cite PMIDs in square brackets after each claim, e.g. [38123456]. "
    "Note when evidence is preliminary (cell lines, mouse models, small studies) "
    "or when studies disagree. Write 120-160 words, no preamble."
)

def get_abstracts(canonical, rel_type, k=8):
    docs = con.execute(f"""
        SELECT a.pmid, t.year, a.title, a.abstract
        FROM topic_pmids t JOIN articles a USING (pmid)
        WHERE t.canonical = ? AND t.rel_type = ?
        ORDER BY t.year DESC, a.pmid DESC
        LIMIT {k}
    """, [canonical, rel_type]).df()
    text = "\n\n".join(f"[PMID {r.pmid}] ({r.year}) {r.title}\n{r.abstract}"
                       for r in docs.itertuples())
    return text, docs["pmid"].tolist()

def summarize(canonical, rel_type):
    abstracts, pmids = get_abstracts(canonical, rel_type)
    relation = "as a treatment for" if rel_type == "treat" else "in relation to"
    msg = client.messages.create(
        model=MODEL, max_tokens=500, system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Topic: recent research on {canonical} {relation} colorectal cancer.\n\n{abstracts}"}])
    return msg.content[0].text, pmids

summary, pmids = summarize("fruquintinib (HMPL-013)", "treat")
print(summary)
print("\nSource PMIDs:", pmids)

Fruquintinib, a selective VEGFR inhibitor, shows efficacy for refractory metastatic colorectal cancer (mCRC). In the phase III FRESCO-2 trial, fruquintinib plus best supportive care improved overall survival (7.6 vs. 4.6 months) and progression-free survival (3.7 vs. 1.8 months) compared to placebo in patients who progressed on prior therapies [41647991]. A meta-analysis of 15 studies confirmed significant improvements in PFS (HR=0.30) and OS (HR=0.66) versus placebo, with manageable toxicity [41522768]. Real-world data suggests fruquintinib combined with immunotherapy achieved longer PFS (8.8 months) than chemotherapy alone in third-line treatment [41480520]. Preliminary evidence indicates combining fruquintinib with PD-1 inhibitors and docetaxel warrants further investigation in MSS/pMMR tumors [41480198]. Mechanistically, fruquintinib reduces immunosuppressive myeloid populations in preclinical models [41459512]. Grade ≥3 adverse events occur in approximately 31% of patients, with h

In [23]:
import re, textwrap

print(textwrap.fill(summary, 110))   # full summary, wrapped

# Pull the text of the 8 source abstracts
src = con.execute(f"SELECT pmid, abstract FROM articles WHERE pmid IN ({','.join(map(str, pmids))})").df()
source_text = " ".join(src["abstract"])

# Check 1: every cited PMID must be one of the 8 sources
cited = set(re.findall(r"\b(\d{7,9})\b", summary))
print("\nCited PMIDs not in sources:", cited - set(map(str, pmids)) or "none")

# Check 2: every number in the summary should appear in some source abstract
nums = set(re.findall(r"\d+(?:\.\d+)?", summary)) - cited
print("Numbers not found in any source:", [n for n in nums if n not in source_text] or "none")

# Show which abstract(s) mention the survival figures
for r in src.itertuples():
    if "7.6" in r.abstract or "4.6" in r.abstract:
        print(f"\n[PMID {r.pmid}]")
        print(textwrap.fill(r.abstract, 110))

Fruquintinib, a selective VEGFR inhibitor, shows efficacy for refractory metastatic colorectal cancer (mCRC).
In the phase III FRESCO-2 trial, fruquintinib plus best supportive care improved overall survival (7.6 vs. 4.6
months) and progression-free survival (3.7 vs. 1.8 months) compared to placebo in patients who progressed on
prior therapies [41647991]. A meta-analysis of 15 studies confirmed significant improvements in PFS (HR=0.30)
and OS (HR=0.66) versus placebo, with manageable toxicity [41522768]. Real-world data suggests fruquintinib
combined with immunotherapy achieved longer PFS (8.8 months) than chemotherapy alone in third-line treatment
[41480520]. Preliminary evidence indicates combining fruquintinib with PD-1 inhibitors and docetaxel warrants
further investigation in MSS/pMMR tumors [41480198]. Mechanistically, fruquintinib reduces immunosuppressive
myeloid populations in preclinical models [41459512]. Grade ≥3 adverse events occur in approximately 31% of
patients, with h

In [24]:
import re, time, textwrap, pandas as pd

SYSTEM = (
    "You summarize recent biomedical research for a non-specialist audience. "
    "Use ONLY the abstracts provided; do not add outside knowledge. "
    "Cite PMIDs in square brackets after each claim, e.g. [38123456]. "
    "Copy all numbers exactly as written in the abstracts. "
    "If a result comes from a subgroup, regional, secondary, or pooled analysis, "
    "say so explicitly (e.g., 'in the Spanish subgroup of FRESCO-2'). "
    "Note when evidence is preliminary (cell lines, mouse models, small studies) "
    "or when studies disagree. Write 120-160 words, no preamble."
)
PROMPT_VERSION = "v2"

WORDS = {w: str(i) for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen "
    "fourteen fifteen sixteen seventeen eighteen nineteen twenty".split())}
NUM = r"\d+(?:\.\d+)?"

def check(summary, pmids):
    text = " ".join(con.execute(
        f"SELECT abstract FROM articles WHERE pmid IN ({','.join(map(str, pmids))})"
    ).df()["abstract"]).lower()
    src_nums = set(re.findall(NUM, text)) | {d for w, d in WORDS.items() if re.search(rf"\b{w}\b", text)}
    cited = set(re.findall(r"\b\d{7,9}\b", summary))
    bad_cites = cited - set(map(str, pmids))
    missing = set(re.findall(NUM, summary)) - cited - src_nums
    return sorted(bad_cites), sorted(missing)

topics = con.sql("SELECT canonical, rel_type FROM selected").df()
results = []
for t in topics.itertuples():
    text_, pmids_ = summarize(t.canonical, t.rel_type)
    bad_cites, missing = check(text_, pmids_)
    results.append({"canonical": t.canonical, "rel_type": t.rel_type, "summary": text_,
                    "source_pmids": ",".join(map(str, pmids_)),
                    "bad_citations": ",".join(bad_cites), "unmatched_numbers": ",".join(missing),
                    "passed_checks": not bad_cites and not missing,
                    "model": MODEL, "prompt_version": PROMPT_VERSION,
                    "created_at": pd.Timestamp.now()})
    print(f"{'PASS' if results[-1]['passed_checks'] else 'FLAG'}  {t.canonical}")
    time.sleep(0.5)

summaries = pd.DataFrame(results)
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries")
print(f"\n{summaries['passed_checks'].sum()} of {len(summaries)} summaries passed both checks")

# Did the new rule fix fruquintinib?
print("\n" + textwrap.fill(summaries.loc[summaries.canonical.str.startswith("fruquintinib"), "summary"].iloc[0], 110))

PASS  N6-methyladenosine (m6A)
PASS  GPX4
PASS  fruquintinib (HMPL-013)
FLAG  encorafenib
PASS  METTL3
PASS  STING1
PASS  Ipilimumab
FLAG  Nivolumab
PASS  PDCD1
PASS  Cholesterol
PASS  KRAS p.G12C
PASS  SLC7A11
FLAG  CD274
FLAG  PIK3CD
PASS  pembrolizumab

11 of 15 summaries passed both checks

# Fruquintinib for Refractory Metastatic Colorectal Cancer  Fruquintinib, a selective VEGFR inhibitor, shows
efficacy for treating refractory metastatic colorectal cancer (mCRC). In the phase III FRESCO-2 trial's
Spanish subgroup (26% of global population), fruquintinib plus best supportive care improved median overall
survival to 7.6 versus 4.6 months versus placebo (HR 0.63) and median progression-free survival to 3.7 versus
1.8 months (HR 0.24) [41647991]. A meta-analysis of 12 studies (n=3,703) confirmed fruquintinib improved PFS
(HR 0.30) and OS (HR 0.66) versus placebo, with objective response rate of 4.9% and disease control rate of
62.2% [41522768]. Grade ≥3 adverse events occurred in 30

In [25]:
NUM = r"(?<![A-Za-z\d.])\d+(?:\.\d+)?(?![A-Za-z\d])"   # standalone numbers only (skips CD274, V600E)

def normalize(t):
    return re.sub(r"(?<=\d),(?=\d{3}\b)", "", t.lower())   # 3,703 -> 3703

def check(summary, pmids):
    text = normalize(" ".join(con.execute(
        f"SELECT abstract FROM articles WHERE pmid IN ({','.join(map(str, pmids))})"
    ).df()["abstract"]))
    s = normalize(summary)
    src_nums = set(re.findall(NUM, text))
    src_nums |= {d for w, d in WORDS.items() if re.search(rf"\b{w}\b", text)}
    src_nums |= {d for r, d in {"i": "1", "ii": "2", "iii": "3", "iv": "4"}.items()
                 if re.search(rf"phase\s*{r}\b", text)}
    cited = set(re.findall(r"\b\d{7,9}\b", s))
    bad_cites = cited - set(map(str, pmids))
    missing = set(re.findall(NUM, s)) - cited - src_nums
    return sorted(bad_cites), sorted(missing)

# Rescore without new API calls
for i, r in summaries.iterrows():
    bad, miss = check(r.summary, [int(p) for p in r.source_pmids.split(",")])
    summaries.at[i, "bad_citations"] = ",".join(bad)
    summaries.at[i, "unmatched_numbers"] = ",".join(miss)
    summaries.at[i, "passed_checks"] = not bad and not miss
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries")
print(f"{summaries['passed_checks'].sum()} of {len(summaries)} passed after fixing the checker\n")

# For anything still flagged, show the exact sentence containing each problem number
for r in summaries[~summaries.passed_checks].itertuples():
    print(f"=== {r.canonical} ===  bad citations: {r.bad_citations or 'none'} | unmatched: {r.unmatched_numbers}")
    for n in filter(None, r.unmatched_numbers.split(",")):
        for sent in re.split(r"(?<=[.!?])\s+", r.summary):
            if re.search(rf"(?<![\d.]){re.escape(n)}(?![\d.])", sent):
                print(f"  [{n}] {sent}")
    print()

13 of 15 passed after fixing the checker

=== Nivolumab ===  bad citations: none | unmatched: 82
  [82] In microsatellite instability-high/mismatch repair-deficient (MSI-H/dMMR) metastatic colorectal cancer, nivolumab plus ipilimumab achieved superior outcomes: 54% objective response rate, 84% overall survival rate, and 82% disease control rate [41237182].

=== PIK3CD ===  bad citations: none | unmatched: 2025
  [2025] # PIK3CD and Colorectal Cancer Research

Recent 2025 studies reveal limited direct PIK3CD investigation, but extensive PI3K/AKT pathway research in colorectal cancer.



In [26]:
r = summaries[summaries.canonical == "Nivolumab"].iloc[0]
src = con.execute(f"SELECT pmid, abstract FROM articles WHERE pmid IN ({r.source_pmids})").df()
found = False
for s in src.itertuples():
    for m in re.finditer(r"disease control|DCR", s.abstract, flags=re.I):
        found = True
        print(f"[PMID {s.pmid}] ...{s.abstract[max(0, m.start() - 150): m.end() + 150]}...\n")
print("Mentions found" if found else "No source mentions disease control -> the 82% was not in the abstracts")

[PMID 41237182] ...across eight databases was conducted by 12 independent reviewers. Extracted outcomes included overall survival (OS), progression-free survival (PFS), disease control rate (DCR), and objective response rate (ORR). To facilitate pooled analysis, data reported as median and interquartile range (IQR), or median, minim...

[PMID 41237182] ... was conducted by 12 independent reviewers. Extracted outcomes included overall survival (OS), progression-free survival (PFS), disease control rate (DCR), and objective response rate (ORR). To facilitate pooled analysis, data reported as median and interquartile range (IQR), or median, minimum, and ma...

[PMID 41237182] ...ficacy across all endpoints: ORR 0.54 [95% CI: 0.45-0.65, I² = 75%], OS 0.84 [95% CI: 0.81-0.88, I² = 0%], PFS 0.73 [95% CI: 0.68-0.78, I² = 0%], and DCR 0.82 [95% CI: 0.77-0.86, I² = 0%]. This combination outperformed NIV alone, which demonstrated ORR 0.36 [95% CI: 0.21-0.60, I² = 81%], OS 0.73 [95% C...

[PMID 4

In [27]:
SYSTEM = (
    "You summarize recent biomedical research for a non-specialist audience. "
    "Use ONLY the abstracts provided; do not add outside knowledge. "
    "Cite PMIDs in square brackets after each claim, e.g. [38123456]. "
    "Report only numbers stated in the abstracts, copied exactly; never calculate, "
    "round, or combine numbers. "
    "If a result comes from a subgroup, regional, secondary, or pooled analysis, say so explicitly. "
    "If the abstracts mostly discuss a broader topic than the one named, say so in the first sentence. "
    "Note when evidence is preliminary (cell lines, mouse models, small studies) or when studies disagree. "
    "Plain text only: no headings, titles, bullet points, or markdown. 120-160 words."
)
PROMPT_VERSION = "v3"

_base_check = check
def check(summary, pmids):
    bad, miss = _base_check(summary, pmids)
    years = {str(y) for (y,) in con.execute(
        f"SELECT DISTINCT year FROM pmid_years WHERE pmid IN ({','.join(map(str, pmids))})").fetchall()}
    return bad, sorted(set(miss) - years)

results = []
for t in con.sql("SELECT canonical, rel_type FROM selected").df().itertuples():
    text_, pmids_ = summarize(t.canonical, t.rel_type)
    bad_cites, missing = check(text_, pmids_)
    results.append({"canonical": t.canonical, "rel_type": t.rel_type, "summary": text_,
                    "source_pmids": ",".join(map(str, pmids_)),
                    "bad_citations": ",".join(bad_cites), "unmatched_numbers": ",".join(missing),
                    "passed_checks": not bad_cites and not missing,
                    "model": MODEL, "prompt_version": PROMPT_VERSION,
                    "created_at": pd.Timestamp.now()})
    print(f"{'PASS' if results[-1]['passed_checks'] else 'FLAG'}  {t.canonical}"
          + (f"  -> {missing}" if missing else ""))
    time.sleep(0.5)

summaries = pd.DataFrame(results)
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries")
print(f"\nv3: {summaries['passed_checks'].sum()} of {len(summaries)} passed both checks")
print("\n" + textwrap.fill(summaries.loc[summaries.canonical == "PIK3CD", "summary"].iloc[0], 110))

PASS  N6-methyladenosine (m6A)
PASS  GPX4
PASS  fruquintinib (HMPL-013)
PASS  encorafenib
PASS  METTL3
PASS  STING1
PASS  Ipilimumab
PASS  Nivolumab
PASS  PDCD1
PASS  Cholesterol
PASS  KRAS p.G12C
PASS  SLC7A11
PASS  CD274
PASS  PIK3CD
PASS  pembrolizumab

v3: 15 of 15 passed both checks

These abstracts discuss colorectal cancer mechanisms and treatments broadly, with limited direct PIK3CD
investigation. While several studies mention the PI3K/AKT pathway, only one provides specific mechanistic
detail: MARCH8 ubiquitinates CEMIP to induce ferroptosis through PI3K/AKT pathway inactivation [40107011].
Additional research identifies PI3K pathway alterations as therapeutic targets. A comprehensive genomic screen
found distinct genetic dependencies interacting with PI3K pathways in colon cancer organoids [40082551].
Network pharmacology analysis revealed the PI3K-Akt pathway among key signaling pathways implicated in
britannin's anti-colorectal cancer effects [40080155]. KIF3C promotes colo

In [28]:
import numpy as np
from sentence_transformers import SentenceTransformer

corpus = con.sql("SELECT pmid, title, abstract FROM articles ORDER BY pmid").df()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb = embedder.encode((corpus["title"] + ". " + corpus["abstract"]).tolist(),
                      normalize_embeddings=True, batch_size=64, show_progress_bar=True)

np.save("data/embeddings.npy", emb)                      # saved so you never re-encode
np.save("data/embedding_pmids.npy", corpus["pmid"].to_numpy())
print(emb.shape)   # (number of abstracts, 384)

c:\Users\aaron\Documents\Github\pubtator-trend-summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aaron\Documents\Github\pubtator-trend-summarizer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aaron\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Devel

(1332, 384)


In [31]:
RAG_SYSTEM = (
    "You answer questions about recent colorectal cancer research using ONLY the abstracts provided. "
    "Cite PMIDs in square brackets after each claim. "
    "Report only numbers stated in the abstracts, copied exactly; never calculate or convert them. "
    "Explicitly label findings from cell-line, animal, subgroup, or small studies as such. "
    "If the abstracts do not contain enough information to answer, say that clearly instead of guessing. "
    "Write one or two paragraphs of plain prose. Do not use headings, bullet points, "
    "or bold or italic text. 150 words or fewer."
)

con.execute("""
CREATE TABLE IF NOT EXISTS rag_log (
    question VARCHAR, answer VARCHAR, retrieved_pmids VARCHAR,
    top_score DOUBLE, passed_checks BOOLEAN, model VARCHAR, created_at TIMESTAMP)
""")

def retrieve(question, k=6):
    q = embedder.encode([question], normalize_embeddings=True)
    scores = (emb @ q.T).ravel()
    top = scores.argsort()[::-1][:k]
    hits = corpus.iloc[top].copy()
    hits["score"] = scores[top].astype(float).round(3)
    return hits

def ask(question, k=6):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[PMID {r.pmid}] {r.title}\n{r.abstract}" for r in hits.itertuples())
    msg = client.messages.create(
        model=MODEL, max_tokens=500, system=RAG_SYSTEM,
        messages=[{"role": "user", "content": f"Question: {question}\n\nAbstracts:\n{context}"}])
    answer = msg.content[0].text
    bad, miss = check(answer, hits["pmid"].tolist())
    passed = not bad and not miss
    con.execute("INSERT INTO rag_log VALUES (?, ?, ?, ?, ?, ?, current_timestamp)",
                [question, answer, ",".join(map(str, hits.pmid)), float(hits.score.iloc[0]), passed, MODEL])
    print(textwrap.fill(answer, 110))
    print("\nRetrieved:", ", ".join(f"{p} ({s:.3f})" for p, s in zip(hits.pmid, hits.score)))
    print("Checks:", "PASS" if passed else f"FLAG  bad_cites={bad}  unmatched={miss}")
    return answer, hits

In [32]:
questions = [
    "What resistance mechanisms to KRAS G12C inhibitors have been reported in colorectal cancer?",
    "How are researchers trying to use ferroptosis against colorectal cancer?",
    "What is the recommended daily aspirin dose for preventing colorectal cancer?",
]
for q in questions:
    print("Q:", q, "\n")
    ask(q)
    print("\n" + "-" * 110 + "\n")

Q: What resistance mechanisms to KRAS G12C inhibitors have been reported in colorectal cancer? 

In colorectal cancer, resistance to KRAS G12C inhibitors involves multiple mechanisms. EGFR reactivation has
emerged as a principal resistance mechanism to KRAS G12C inhibitor monotherapy [38825433]. Additionally, the
abstracts indicate that resistance is associated with acquired genetic alterations, reactivation of downstream
pathways, and stimulation of upstream signaling [36632627]. Alternative RTK (receptor tyrosine kinase)
reactivation and acquired molecular resistance mechanisms have also been documented [38825433]. Primary
resistance also contributes to the limited efficacy of monotherapy in colorectal cancer, and the field has
consequently shifted toward combination strategies, particularly dual EGFR/KRAS blockade, which has
demonstrated promising efficacy [38825433]. The complexity and diversity of drug resistance mechanisms in
tumor cells remains incompletely understood [39956882]

In [33]:
print("RSL3" in con.execute("SELECT abstract FROM articles WHERE pmid = 39895914").fetchone()[0])

False


In [34]:
import os
os.makedirs("app", exist_ok=True)

con.execute("""
CREATE OR REPLACE TABLE topic_year_counts AS
SELECT s.rel_type, s.kind, s.canonical, y.year, COUNT(DISTINCT r.pmid) AS n
FROM selected s
JOIN partner_map m ON m.canonical = s.canonical
JOIN crc_relations r
  ON r.rel_type = s.rel_type
 AND m.partner = CASE WHEN r.entity1 = 'Disease|MESH:D015179' THEN r.entity2 ELSE r.entity1 END
JOIN pmid_years y ON y.pmid = r.pmid
WHERE y.year BETWEEN 2015 AND 2025
GROUP BY ALL
""")

con.execute("""
CREATE OR REPLACE TABLE field_year_counts AS
SELECT y.year, COUNT(DISTINCT r.pmid) AS n
FROM crc_relations r JOIN pmid_years y USING (pmid)
WHERE y.year BETWEEN 2015 AND 2025
GROUP BY 1 ORDER BY 1
""")

con.execute("ATTACH 'app/app_data.duckdb' AS app")
for t in ["selected", "topic_year_counts", "field_year_counts", "summaries", "rag_log"]:
    con.execute(f"CREATE OR REPLACE TABLE app.{t} AS SELECT * FROM {t}")
con.execute("CREATE OR REPLACE TABLE app.article_titles AS SELECT pmid, title FROM articles")
con.execute("DETACH app")

print(con.sql("SELECT COUNT(*) AS rows FROM topic_year_counts"))
print(round(os.path.getsize("app/app_data.duckdb") / 1e6, 1), "MB")

┌───────┐
│ rows  │
│ int64 │
├───────┤
│   141 │
└───────┘

1.8 MB


In [35]:
print("In memory:", summaries["prompt_version"].unique())   # should print ['v3']

con.register("summaries_v3", summaries)
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries_v3")
con.unregister("summaries_v3")
print(con.sql("SELECT prompt_version, COUNT(*) AS n, SUM(passed_checks::INT) AS passed FROM summaries GROUP BY 1"))

con.execute("ATTACH 'app/app_data.duckdb' AS app")
con.execute("CREATE OR REPLACE TABLE app.summaries AS SELECT * FROM summaries")
con.execute("DETACH app")

In memory: <StringArray>
['v3']
Length: 1, dtype: str
┌────────────────┬───────┬────────┐
│ prompt_version │   n   │ passed │
│    varchar     │ int64 │ int128 │
├────────────────┼───────┼────────┤
│ v3             │    15 │     15 │
└────────────────┴───────┴────────┘



In [36]:
import requests, time
ESUM = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
EXCLUDE = {"Retracted Publication", "Retraction of Publication", "Published Erratum", "Expression of Concern"}

art_pmids = con.sql("SELECT pmid FROM articles").df()["pmid"].astype(str).tolist()
flagged = {}
for i in range(0, len(art_pmids), 200):
    res = requests.post(ESUM, data={"db": "pubmed", "retmode": "json", "email": "aaronpong21@gmail.com",
                                    "id": ",".join(art_pmids[i:i + 200])}, timeout=60).json()["result"]
    for uid in res.get("uids", []):
        hit = set(res[uid].get("pubtype", [])) & EXCLUDE
        if hit:
            flagged[int(uid)] = ", ".join(sorted(hit))
    time.sleep(0.35)

# Backup check on titles
for p in con.sql("""SELECT pmid FROM articles WHERE regexp_matches(title,
        '^(retraction|retracted|correction|erratum|expression of concern)', 'i')""").df()["pmid"]:
    flagged.setdefault(int(p), "title")

print(f"{len(flagged)} of {len(art_pmids)} articles flagged\n")
if flagged:
    print(con.execute(f"SELECT pmid, title FROM articles WHERE pmid IN ({','.join(map(str, flagged))})").df().to_string())

5 of 1332 articles flagged

       pmid                                                                                                                                                                                       title
0  39905022                                                                Rechallenge therapy versus tyrosine kinase inhibitor (TKI) for advanced metastatic colorectal cancer: a retrospective study.
1  37336360                                                                                         Dual targeting of TGF-β and PD-L1 inhibits tumor growth in TGF-β/PD-L1-driven colorectal carcinoma.
2  37503727                                                                                     [Corrigendum] METTL3 antagonizes 5‑FU chemotherapy and confers drug resistance in colorectal carcinoma.
3  41133968  RETRACTION: Regulatory Mechanism of RNA Binding Motif Protein 15-Mediated N<sup>6</sup> Methyladenosine Modification in Proliferation, Invasion, and Migration 

In [37]:
import numpy as np, pandas as pd

if flagged:
    con.execute(f"DELETE FROM articles WHERE pmid IN ({','.join(map(str, flagged))})")
print(con.sql("SELECT COUNT(*) AS articles_left FROM articles"))

# 1) Re-summarize all 15 topics: same v3 prompt, cleaned sources
results = []
for t in con.sql("SELECT canonical, rel_type FROM selected").df().itertuples():
    text_, pmids_ = summarize(t.canonical, t.rel_type)
    bad_cites, missing = check(text_, pmids_)
    results.append({"canonical": t.canonical, "rel_type": t.rel_type, "summary": text_,
                    "source_pmids": ",".join(map(str, pmids_)),
                    "bad_citations": ",".join(bad_cites), "unmatched_numbers": ",".join(missing),
                    "passed_checks": not bad_cites and not missing,
                    "model": MODEL, "prompt_version": "v3", "created_at": pd.Timestamp.now()})
    time.sleep(0.5)
summaries = pd.DataFrame(results)
con.register("summaries_df", summaries)
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries_df")
con.unregister("summaries_df")
print(f"Summaries: {summaries['passed_checks'].sum()} of {len(summaries)} passed checks\n")

# 2) Rebuild the RAG index on the cleaned corpus and re-ask the test questions
corpus = con.sql("SELECT pmid, title, abstract FROM articles ORDER BY pmid").df()
emb = embedder.encode((corpus["title"] + ". " + corpus["abstract"]).tolist(),
                      normalize_embeddings=True, batch_size=64, show_progress_bar=True)
np.save("data/embeddings.npy", emb)
np.save("data/embedding_pmids.npy", corpus["pmid"].to_numpy())

con.execute("DELETE FROM rag_log")
for q in ["What resistance mechanisms to KRAS G12C inhibitors have been reported in colorectal cancer?",
          "How are researchers trying to use ferroptosis against colorectal cancer?",
          "What is the recommended daily aspirin dose for preventing colorectal cancer?"]:
    print("Q:", q); ask(q); print()

# 3) Re-export for the dashboard
con.execute("ATTACH 'app/app_data.duckdb' AS app")
for t in ["summaries", "rag_log"]:
    con.execute(f"CREATE OR REPLACE TABLE app.{t} AS SELECT * FROM {t}")
con.execute("CREATE OR REPLACE TABLE app.article_titles AS SELECT pmid, title FROM articles")
_ = con.execute("DETACH app")

┌───────────────┐
│ articles_left │
│     int64     │
├───────────────┤
│          1327 │
└───────────────┘

Summaries: 14 of 15 passed checks



Batches: 100%|██████████| 21/21 [00:21<00:00,  1.02s/it]


Q: What resistance mechanisms to KRAS G12C inhibitors have been reported in colorectal cancer?
The provided abstracts describe resistance mechanisms to KRAS G12C inhibitors in colorectal cancer but do not
detail specific molecular mechanisms with precise citations. The abstracts indicate that resistance involves
acquired genetic alterations, reactivation of downstream pathways, and stimulation of upstream signaling [PMID
36632627]. EGFR reactivation has emerged as a principal resistance mechanism to KRAS inhibitor monotherapy
[PMID 38825433]. Additionally, alternative RTK reactivation and acquired molecular resistance mechanisms have
been identified [PMID 38825433]. The abstracts note that primary resistance also occurs, with limited coverage
of prevalent KRAS mutations such as G12D and G12V [PMID 40692745]. However, these abstracts provide only
general descriptions of resistance categories rather than specific mechanistic details with quantified
findings that could be cited individual

In [38]:
import re
for p, why in flagged.items():
    print(p, "->", why)
print()

for r in summaries[~summaries["passed_checks"]].itertuples():
    print(f"=== {r.canonical} ===  bad citations: {r.bad_citations or 'none'} | unmatched: {r.unmatched_numbers}")
    for n in filter(None, r.unmatched_numbers.split(",")):
        for sent in re.split(r"(?<=[.!?])\s+", r.summary):
            if re.search(rf"(?<![\d.]){re.escape(n)}(?![\d.])", sent):
                print(f"  [{n}] {sent}")

39905022 -> Retracted Publication
37336360 -> Retracted Publication
37503727 -> Published Erratum
37471637 -> Retracted Publication
41133968 -> title

=== KRAS p.G12C ===  bad citations: none | unmatched: 20
  [20] Among KRAS subtypes, G12D (30.37%), G13D (20%), and G12V (17.78%) were most common in one cohort [41365930].


In [39]:
ab = con.execute("SELECT abstract FROM articles WHERE pmid = 41365930").fetchone()[0]
i = ab.find("G13D")
print(ab[max(0, i - 150): i + 150])

value was evaluated using the Cox proportional hazards model. The KRAS mutation rate was 52.7% (135/256), with the main subtypes being G12D (30.37%), G13D (20.00%), G12V (17.78%), and G12C (7.41%). Logistic regression showed that KRAS mutation independently predicted right-sided colon tumors (OR = 1


In [40]:
def normalize(t):
    t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t.lower())   # 3,703 -> 3703
    t = re.sub(r"(\d+\.\d*?[1-9])0+\b", r"\1", t)        # 8.10  -> 8.1
    return re.sub(r"(\d+)\.0+\b", r"\1", t)               # 20.00 -> 20

for i, r in summaries.iterrows():
    bad, miss = check(r.summary, [int(p) for p in r.source_pmids.split(",")])
    summaries.at[i, "bad_citations"] = ",".join(bad)
    summaries.at[i, "unmatched_numbers"] = ",".join(miss)
    summaries.at[i, "passed_checks"] = not bad and not miss
print(f"{summaries['passed_checks'].sum()} of {len(summaries)} passed")

con.register("summaries_df", summaries)
con.execute("CREATE OR REPLACE TABLE summaries AS SELECT * FROM summaries_df")
con.unregister("summaries_df")
con.execute("ATTACH 'app/app_data.duckdb' AS app")
con.execute("CREATE OR REPLACE TABLE app.summaries AS SELECT * FROM summaries")
_ = con.execute("DETACH app")

15 of 15 passed
